# 🎯 Objetivos

El objetivo es construir una base de datos final sobre la que entrenar el sistema de recomendación, aplicando las transformaciones necesarias basadas en el EDA y generando las representaciones numéricas necesarias para el modelo basado en contenido.

* ⚙️ **Configuración inicial**. Rutas, paquetes y funciones.
* ♻️ **Carga de datos**.
* 🧹 **Filtrado inicial de los datos**. Limpieza inicial de filas y columnas.
* 🏷️ **Tratamiento de las variables categóricas**.
* 🔘 **Tratamiento de las variables booleanas**.
* 🕸️ **Vectorización TF-IDF**. Tratamiento de la variable textual text_full_clean. Experimentar varios hiperparámetros y entrenar.
* 🧊 **Creación de la matriz estructural**.
* 💾 **Guardado de información**. Guardado de información y resumen de los datos almacenados.

# ⚙️ Configuración

## Rutas (paths)

In [1]:
import os

# Rutas en función de la carpeta raíz del proyecto (README.md)
base_path = os.path.dirname(os.path.dirname(os.path.abspath(__file__))) if "__file__" in locals() else os.path.abspath("..")

# Subrutas
data_dir = os.path.join(base_path, "data")
db_path = os.path.join(data_dir, "steam.db")
src_dir = os.path.join(base_path, "src")
models_dir = os.path.join(base_path, "models")


## Paquetes y funciones

In [2]:
# Sistema y configuración
import warnings
warnings.filterwarnings("ignore")
import sqlite3
import itertools
import joblib
import json

# Análisis y manipulación de datos
import pandas as pd
import numpy as np

# ML y procesamiento
from sklearn.preprocessing import MinMaxScaler, normalize
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity

# Matrices dispersas
from scipy.sparse import csr_matrix
from scipy import sparse  

# ♻️ Carga de datos

In [3]:
# Información del EDA
conn = sqlite3.connect(db_path)
cur = conn.cursor()

games_fe_raw = pd.read_sql("SELECT * FROM games_eda", conn)

conn.close()

# Hacemos una copia para trabajar
games_fe_raw = games_fe_raw.copy()

# 🧹 Filtrado inicial de los datos

Conservar solo las columnas relevantes y las filas con información suficiente.

In [4]:
# Eliminar columnas no necesarias
cols_to_remove = [
    "description", "content_descriptors_notes", "genres_total", "text_full",
    "description_clean", "content_clean", "genres_clean",
    "description_clean_len", "content_clean_len", "genres_clean_len", "text_full_clean_len",
    "positive_reviews", "total_reviews", "average_forever", "median_forever","windows",
    "release_date", "developers", "publishers", "lang_detected"
]

# Añadir todas las columnas de géneros (genre_*) para eliminarlas
genre_cols = [c for c in games_fe_raw.columns if c.startswith("genre_")]
cols_to_remove.extend(genre_cols)

# Filtrar solo columnas que realmente existen en el dataframe
cols_to_remove = [c for c in cols_to_remove if c in games_fe_raw.columns]

# Eliminar columnas
games_fe = games_fe_raw.drop(columns=cols_to_remove).copy()

print("Columnas eliminadas:", len(cols_to_remove))
print("Forma final del dataframe:", games_fe.shape)

Columnas eliminadas: 53
Forma final del dataframe: (2475, 13)


In [5]:
# Eliminar los que no tengan texto o sea NaN
games_fe["text_full_clean"] = games_fe["text_full_clean"].fillna("").str.strip()
games_fe = games_fe[games_fe["text_full_clean"] != ""].copy()

# Filtrar texto corto (muy ruidoso)
games_fe["len_text"] = games_fe["text_full_clean"].str.split().apply(len)
games_fe = games_fe[games_fe["len_text"] > 20].copy()
games_fe = games_fe.drop(columns=["len_text"])

# Eliminar filas con algún vacío --> soluciona problemas en X_struct
games_fe = games_fe.dropna(how="any").copy()

print("Filas tras eliminar NaN:", games_fe.shape)

Filas tras eliminar NaN: (2468, 13)


In [6]:
# Revisión duplicados
assert games_fe["appid"].is_unique, "Hay appid duplicados"

# 🏷️ Tratamiento variables categóricas

En este aparado transformamos la variable categórica developer_level en tres variables booleanas (codificación one-hot).

In [7]:
# One-hot encoding
dummies = pd.get_dummies(games_fe["developer_level"], prefix="developer")
games_fe = pd.concat([games_fe, dummies], axis=1)
games_fe = games_fe.drop(columns=["developer_level"])

# Columnas resultantes
print("One-hot developer_level aplicado:", list(dummies.columns))

One-hot developer_level aplicado: ['developer_large', 'developer_medium', 'developer_small']


# 🔘 Tratamiento variables booleanas

En este apartado se normaliza el formato de las variables booleanas para poder usarlo en el sistema de recomendación. Se comprueba que no existan valores nulos y se fuerza a que sean de tipo entero (int64) para poder escalar y combinarse con el resto de variables en la matriz estructural.

In [8]:
# Variables booleanas
bool_cols  = [
    "free", "mac", "linux", "multijugador", "cooperativo", "competitivo", "multidispositivo",
    "developer_large", "developer_medium", "developer_small"
    ]

# Asegurar que son enteros
for c in bool_cols:
    games_fe[c] = games_fe[c].fillna(0).astype(int)

# 🕸️ Vectorización TF-IDF

## Experimento hiperparámetros

Probar diferentes hiperparámetros para encontrar la mejor configuración.

In [9]:
# Añadir stopwords específicas de videojuegos (mejora rendimiento)
extra_stopwords = [
    "game", "games", "player", "players", "world", "experience",
    "system", "play", "played", "gameplay", "level", "levels",
    "mode", "modes", "feature", "features", "vision", "emphasize",
    "model", "new", "character", "unique", "time", "include"
]

stopwords_total = list(ENGLISH_STOP_WORDS.union(extra_stopwords))

In [10]:
# Semilla para reproducibilidad
np.random.seed(42)

# Parámetros a probar
param_grid = {
    "max_features":    [10000, 15000, 20000],
    "ngram_range":     [(1,1), (1,2)],
    "min_df":          [3, 5],
    "max_df":          [0.75, 0.85],
    "sublinear_tf":    [True, False],   
    "stop_words":      [stopwords_total], 
    "norm":            ["l2"], 
}

# Combinaciones
keys = list(param_grid.keys())
values_product = list(itertools.product(*param_grid.values()))

print(f"Nº configuraciones a probar: {len(values_product)}\n")

results = []
configs = []
sample_text = games_fe["text_full_clean"].fillna("")

for values in values_product:
    cfg = dict(zip(keys, values))
    configs.append(cfg)

    try:
        tfidf_tmp = TfidfVectorizer(**cfg)
        tfidf_matrix_tmp = tfidf_tmp.fit_transform(sample_text)

        # Matriz de similitud coseno completa (n_juegos x n_juegos)
        sim_matrix = cosine_similarity(tfidf_matrix_tmp)

        # Métricas
        sim_mean = np.mean(sim_matrix)
        sim_std = np.std(sim_matrix)

        # Top-5 similarity (excluyendo el propio juego)
        sim_no_diag = sim_matrix - np.eye(sim_matrix.shape[0])
        top5_sim_mean = np.mean(np.sort(sim_no_diag, axis=1)[:, -5:])

        sparsity = 1.0 - (
            tfidf_matrix_tmp.nnz / (tfidf_matrix_tmp.shape[0] * tfidf_matrix_tmp.shape[1])
        )

        # Métrica compuesta: cuánto destacan los 5 vecinos más similares frente a la similitud media global
        top5_over_mean = top5_sim_mean / (sim_mean + 1e-8)

        results.append({
            "max_features": cfg["max_features"],
            "ngram_range": cfg["ngram_range"],
            "min_df": cfg["min_df"],
            "max_df": cfg["max_df"],
            "sublinear_tf": cfg["sublinear_tf"],
            "mean_similarity": sim_mean,
            "std_similarity": sim_std,
            "top5_similarity": top5_sim_mean,
            "top5_over_mean": top5_over_mean,
            "sparsity": sparsity
        })

    except Exception as e:
        print(f"Error en la configuración {cfg}: {e}")

# DataFrame de resultados del experimento
results_df = pd.DataFrame(results)
print("\nTop 10 configuraciones ordenadas por calidad de vecinos (top5_over_mean):")
if not results_df.empty:
    display(
        results_df.sort_values(
            ["top5_over_mean", "top5_similarity", "std_similarity"],
            ascending=[False, False, False]
        ).head(10)
    )
else:
    raise RuntimeError("No se ha podido entrenar ningún modelo TF-IDF en el experimento.")

# Guardar resultados del experimento
results_path = os.path.join(models_dir, "tfidf_experiment_results.csv")
results_df.to_csv(results_path, index=False)
print(f"Resultados del experimento guardados en: {results_path}")

# Seleccionar configuración óptima. Priorización de vecino sismilares
best_idx = results_df.sort_values(
    ["top5_over_mean", "top5_similarity", "std_similarity"],
    ascending=[False, False, False]
).index[0]
best_model = results_df.loc[best_idx]
best_params = {
    "max_features": best_model["max_features"],
    "ngram_range": best_model["ngram_range"],
    "min_df": best_model["min_df"],
    "max_df": best_model["max_df"],
    "sublinear_tf": best_model["sublinear_tf"],
    "stop_words": stopwords_total,  
    "norm": "l2",                       
}

print("\nMejor configuración:")
print(best_model)

Nº configuraciones a probar: 48


Top 10 configuraciones ordenadas por calidad de vecinos (top5_over_mean):


,max_features,ngram_range,min_df,max_df,sublinear_tf,mean_similarity,std_similarity,top5_similarity,top5_over_mean,sparsity
41,20000,"(1, 2)",3,0.75,False,0.041573,0.036449,0.234963,5.651814,0.990284
43,20000,"(1, 2)",3,0.85,False,0.041573,0.036449,0.234963,5.651814,0.990284
25,15000,"(1, 2)",3,0.75,False,0.044131,0.038073,0.245054,5.552859,0.987481
27,15000,"(1, 2)",3,0.85,False,0.044131,0.038073,0.245054,5.552859,0.987481
1,10000,"(1, 1)",3,0.75,False,0.044413,0.036551,0.246260,5.544827,0.984976
3,10000,"(1, 1)",3,0.85,False,0.044413,0.036551,0.246260,5.544827,0.984976
17,15000,"(1, 1)",3,0.75,False,0.044413,0.036551,0.246260,5.544827,0.984976
19,15000,"(1, 1)",3,0.85,False,0.044413,0.036551,0.246260,5.544827,0.984976
33,20000,"(1, 1)",3,0.75,False,0.044413,0.036551,0.246260,5.544827,0.984976
35,20000,"(1, 1)",3,0.85,False,0.044413,0.036551,0.246260,5.544827,0.984976


Resultados del experimento guardados en: c:\Users\User\Desktop\system_recommendation_steam\models\tfidf_experiment_results.csv

Mejor configuración:
max_features          20000
ngram_range          (1, 2)
min_df                    3
max_df                 0.75
sublinear_tf          False
mean_similarity    0.041573
std_similarity     0.036449
top5_similarity    0.234963
top5_over_mean     5.651814
sparsity           0.990284
Name: 41, dtype: object


## TF-IDF entrenamiento final

In [11]:
tfidf_final = TfidfVectorizer(**best_params)
X_tfidf = tfidf_final.fit_transform(
    games_fe["text_full_clean"].fillna("")
).astype(np.float32)

joblib.dump(tfidf_final, os.path.join(models_dir, "tfidf_vectorizer.pkl"))
sparse.save_npz(os.path.join(models_dir, "tfidf_matrix.npz"), X_tfidf)

print("TF-IDF final creado:", X_tfidf.shape)

TF-IDF final creado: (2468, 20000)


# 🧊 Matriz estructural

En este apartado se construye la matriz estructural del sistema de recomendación, que recoge las características no textuales de cada juego en formato numérico. Se escalan y normalizan las variables.

In [12]:
# Variables a escalar
cols_princ = [
    "score_reviews", "release_year","multijugador", "cooperativo", "competitivo", "multidispositivo", "free", "mac", "linux",
    "developer_large", "developer_medium", "developer_small"
    ]

In [13]:
# Verificación de que todas existen en el dataframe
missing = [c for c in cols_princ if c not in games_fe.columns]
if missing:
    raise ValueError(f"Faltan columnas en el dataframe para X_struct: {missing}")

In [14]:
X_struct_raw = games_fe[cols_princ].astype(float).copy()

# Escalado conjunto al rango [0,1]. Incluye variables booleanas y release_year
scaler_struct = MinMaxScaler()
X_struct_scaled = scaler_struct.fit_transform(X_struct_raw)

In [15]:
# Normalización por fila (L2)
X_struct_norm = normalize(X_struct_scaled, norm="l2", axis=1)

In [16]:
# Convertir a formato disperso para compatibilidad con TF-IDF
X_struct = csr_matrix(X_struct_norm)

In [17]:
print("Matriz estructural creada:", X_struct.shape)

Matriz estructural creada: (2468, 12)


# 💾 Guardar información

## Modelo y resultados TF-IDF

In [18]:
tfidf_model_path = os.path.join(models_dir, "tfidf_vectorizer.pkl")
joblib.dump(tfidf_final, tfidf_model_path)

tfidf_matrix_path = os.path.join(models_dir, "tfidf_matrix.npz")
sparse.save_npz(tfidf_matrix_path, X_tfidf)

games_index_path = os.path.join(models_dir, "games_index.csv")
games_fe[["appid", "name"]].to_csv(games_index_path, index=False)

## Matriz estructural y escalador

In [19]:
joblib.dump(scaler_struct, os.path.join(models_dir, "scaler_struct.pkl"))
sparse.save_npz(os.path.join(models_dir, "X_struct.npz"), X_struct)

with open(os.path.join(models_dir, "cols_princ.json"), "w") as f:
    json.dump(cols_princ, f)

## Tabla games 

Guardamos la versión sin escalar y la escalda.

In [20]:
conn = sqlite3.connect(db_path)

# df escalado y normalizado
X_struct_dense = X_struct.toarray()
df_scaled = pd.DataFrame(X_struct_dense, columns=cols_princ)

# Añadimos los identificadores
df_scaled.insert(0, "name", games_fe["name"].values)
df_scaled.insert(0, "appid", games_fe["appid"].values)

# Añadimos la columna text_full_clean (no está porque no se ha escalado)
df_scaled["text_full_clean"] = games_fe["text_full_clean"]

# Guardamos la versión final.
df_scaled.to_sql("games_final_scaled", conn, if_exists="replace", index=False)

# Tabla sin escalar.
games_fe.to_sql("games_fe_clean_unscaled", conn, if_exists="replace", index=False)

conn.close()


In [21]:
# Número de filas guardadas
print(f"Número de filas guardadas en la tabla 'games_final_scaled': {len(df_scaled):,}")

# Columnas guardadas
print(f"Columnas guardadas en la tabla 'games_final_scaled': {', '.join(df_scaled.columns)}")

# Número de filas guardadas
print(f"Número de filas guardadas en la tabla 'games_fe_clean_unscaled': {len(games_fe):,}")

# Columnas guardadas
print(f"Columnas guardadas en la tabla 'games_fe_clean_unscaled': {', '.join(games_fe.columns)}")

Número de filas guardadas en la tabla 'games_final_scaled': 2,468
Columnas guardadas en la tabla 'games_final_scaled': appid, name, score_reviews, release_year, multijugador, cooperativo, competitivo, multidispositivo, free, mac, linux, developer_large, developer_medium, developer_small, text_full_clean
Número de filas guardadas en la tabla 'games_fe_clean_unscaled': 2,468
Columnas guardadas en la tabla 'games_fe_clean_unscaled': appid, name, score_reviews, free, mac, linux, multijugador, cooperativo, competitivo, multidispositivo, release_year, text_full_clean, developer_large, developer_medium, developer_small
